In [1]:
import pandas as pd
import numpy as np
import requests
import zipfile
import io

def load_gscpi_data():
    """Fetches the official GSCPI data directly from the NY Fed."""
    print("-> Fetching NY Fed GSCPI...")
    url = "https://www.newyorkfed.org/medialibrary/research/interactives/gscpi/downloads/gscpi_data.xlsx"
    
    try:
        # 1. Read without skiprows so pandas catches the true headers on the first row
        df_gscpi = pd.read_excel(url, sheet_name='GSCPI Monthly Data')
        
        # 2. Dynamically locate the GSCPI column (handles NY Fed format changes)
        gscpi_cols = [col for col in df_gscpi.columns if 'GSCPI' in str(col).upper()]
        if gscpi_cols:
            df_gscpi = df_gscpi[[df_gscpi.columns[0], gscpi_cols[0]]]
        else:
            # Fallback based on your note: Date in col 0, GSCPI in col 3
            df_gscpi = df_gscpi.iloc[:, [0, 3]]
            
        # 3. Rename columns safely
        df_gscpi.columns = ['Date', 'GSCPI_Value']
        
        # 4. Coerce to datetime and numeric. This smartly turns all the text metadata 
        # rows (like "NEW YORK FED ECONOMIC RESEARCH") into NaT and NaN.
        df_gscpi['Date'] = pd.to_datetime(df_gscpi['Date'], errors='coerce')
        df_gscpi['GSCPI_Value'] = pd.to_numeric(df_gscpi['GSCPI_Value'], errors='coerce')
        
        # 5. Drop the NaT/NaN rows to cleanly remove all the junk metadata
        df_gscpi = df_gscpi.dropna(subset=['Date', 'GSCPI_Value'])
        
        # Convert to month-end period for easier merging
        df_gscpi['YearMonth'] = df_gscpi['Date'].dt.to_period('M')
        return df_gscpi
        
    except Exception as e:
        print(f"   [ERROR] GSCPI load failed: {e}")
        return pd.DataFrame()

def load_dataco_pipeline():
    """Loads DataCo from GitHub, creates late delivery target, and merges GSCPI Environment."""
    print("-> Starting DataCo ingestion...")
    
    url = "https://raw.githubusercontent.com/SristiSarkarMCKV/Causal-RL-for-Supply-Chain-Optimization/main/data/raw/DataCoSupplyChainDataset.zip"
    
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()

        # Open zip file
        zip_file = zipfile.ZipFile(io.BytesIO(response.content))

        # Load CSV inside zip
        df = pd.read_csv(zip_file.open('DataCoSupplyChainDataset.csv'), encoding='latin1')
        
        # Ensure dates are formatted and target is clear
        df['shipping_date'] = pd.to_datetime(df['shipping date (DateOrders)'])
        df['late_delivery_risk'] = df['Late_delivery_risk']
        
        # Create a merging key and attach GSCPI (The Environment)
        df['YearMonth'] = df['shipping_date'].dt.to_period('M')
        df_gscpi = load_gscpi_data()
        
        if not df_gscpi.empty:
            df = df.merge(df_gscpi[['YearMonth', 'GSCPI_Value']], on='YearMonth', how='left')
            # Fill missing with a baseline neutral environment (0.0 means normal pressure)
            df['GSCPI_Value'] = df['GSCPI_Value'].fillna(0.0) 
        
        print("   [SUCCESS] DataCo + GSCPI Environment loaded.")
        return df
        
    except Exception as e:
        print(f"   [ERROR] DataCo load failed: {e}")
        return pd.DataFrame()

def load_walmart_pipeline():
    """Loads Walmart data from GitHub and merges DNA + Environment + Causal Factors."""
    print("-> Starting Walmart ingestion...")
    
    url = "https://raw.githubusercontent.com/SristiSarkarMCKV/Causal-RL-for-Supply-Chain-Optimization/main/data/raw/walmart-recruiting-store-sales-forecasting.zip"
    
    try:
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        main_zip_bytes = io.BytesIO(response.content)

        def extract_nested_csv(master_zip, target_name):
            """Helper to extract CSVs, handling both zipped and flat files."""
            if target_name.endswith('.zip'):
                with master_zip.open(target_name) as nested:
                    with zipfile.ZipFile(io.BytesIO(nested.read())) as internal:
                        csv_name = target_name.replace('.zip', '')
                        return pd.read_csv(internal.open(csv_name))
            else:
                return pd.read_csv(master_zip.open(target_name))

        # Extract components
        with zipfile.ZipFile(main_zip_bytes) as master:
            train = extract_nested_csv(master, 'train.csv.zip')       # 421k records
            features = extract_nested_csv(master, 'features.csv.zip') # Causal factors
            stores = extract_nested_csv(master, 'stores.csv')         # Store attributes

        # Merge DNA (Stores) + Sales (Train) + Environment (Features)
        df = train.merge(stores, on='Store', how='left')
        df = df.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')
        
        # Format Date and handle Markdowns (Causal Treatments)
        df['Date'] = pd.to_datetime(df['Date'])
        markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
        df[markdown_cols] = df[markdown_cols].fillna(0)
        
        # Create Causal RL Target: Risk of severe sales drop
        thresholds = df.groupby(['Store', 'Dept'])['Weekly_Sales'].transform(lambda x: x.quantile(0.10))
        df['low_sales_risk'] = (df['Weekly_Sales'] < thresholds).astype(int)
        
        print("   [SUCCESS] Walmart loaded.")
        return df
        
    except Exception as e:
        print(f"   [ERROR] Walmart load failed: {e}")
        return pd.DataFrame()

if __name__ == "__main__":
    # Execute in memory
    df_dataco = load_dataco_pipeline()
    df_walmart = load_walmart_pipeline()

    # Safe shape checking
    dataco_shape = df_dataco.shape if not df_dataco.empty else "Failed"
    walmart_shape = df_walmart.shape if not df_walmart.empty else "Failed"

    print("-" * 100)
    print(f"DataCo Shape: {dataco_shape} | Walmart Shape: {walmart_shape}")

-> Starting DataCo ingestion...
-> Fetching NY Fed GSCPI...
   [SUCCESS] DataCo + GSCPI Environment loaded.
-> Starting Walmart ingestion...
   [SUCCESS] Walmart loaded.
----------------------------------------------------------------------------------------------------
DataCo Shape: (180519, 57) | Walmart Shape: (421570, 17)
